# Camada Silver — V-Commerce CRM 360

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

---

## Visão Geral

A camada Silver é responsável por transformar os dados brutos ingeridos na Bronze em tabelas limpas, padronizadas e semanticamente organizadas segundo a **modelagem dimensional** (estrela). Cada notebook desta camada lê de `bronze.*`, aplica as transformações necessárias e grava em `silver.*`.

Os princípios aplicados em todas as tabelas Silver são:

- **Deduplicação** — reprocessamentos da Bronze (modo `append`) são neutralizados via `row_number()` sobre a chave primária ordenada por `timestamp_ingestion`
- **Tipagem explícita** — colunas lidas como `string` na Bronze recebem o tipo correto (`timestamp`, `decimal`, `boolean`, `int`)
- **Sanitização textual** — valores categóricos inconsistentes (variações de case, abreviações, typos) são normalizados para um conjunto canônico
- **Colunas derivadas** — atributos calculáveis a partir dos dados brutos são materializados para evitar recomputação nas camadas superiores
- **Idempotência** — todas as escritas usam `mode=overwrite`, garantindo que reexecutar o notebook produza sempre o mesmo resultado
- **Rastreabilidade** — `timestamp_ingestion` é recriado com o instante da carga Silver

---

## Tabelas produzidas neste notebook

### Tabelas Fato

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `ft_tickets_suporte` | `suporte_tickets` | Tickets do SAC limpos, tipados e com colunas derivadas |
| `ft_pedidos` | `pedidos` | Histórico de compras com valor total e ano_mes derivados |
| `ft_avaliacoes` | `avaliacoes` | Avaliações pós-compra com categoria NPS derivada |
| `ft_clickstream` | `clickstream` | Eventos de navegação tipados e normalizados |

### Dimensões

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `dim_clientes` | `clientes` | Perfis cadastrais com idade, nome completo e região derivados |
| `dim_produtos` | `catalogo_produtos` | Catálogo de produtos com categoria normalizada |
| `dim_tipos_problema` | `suporte_tickets` | Tipos de problema canônicos com categoria de negócio |
| `dim_agentes_suporte` | `suporte_tickets` | Agentes com métricas agregadas de desempenho |
| `dim_status_pedido` | `pedidos` | Status de pedido normalizados |
| `dim_categorias_produto` | `catalogo_produtos` | Categorias de produto normalizadas |

---

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
catalogo       = 'vcommerce_catalog'
bronze_schema  = 'vcommerce_bronze'
silver_schema  = 'vcommerce_silver'

spark.sql(f'USE CATALOG {catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {silver_schema}')
spark.sql(f'USE SCHEMA {silver_schema}')

print(f'Catálogo : {catalogo}')
print(f'Schema   : {silver_schema}')

Catálogo : vcommerce_catalog
Schema   : vcommerce_silver


---

## Tratamento: `ft_tickets_suporte`

**Origem:** `bronze.suporte_tickets`  
**Destino:** `silver.ft_tickets_suporte`, `silver.dim_tipos_problema`, `silver.dim_agentes_suporte`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `tipo_problema` | ~25 variações para 4 categorias reais (`pro`, `p3oduto`, `PRODUCT`, `DELAY`…) | Mapeamento para valores canônicos |
| `data_resolucao` | 2.072 nulos — tickets ainda abertos | Mantidos como `null`; `resolvido = false` |
| `nota_avaliacao` | 2.072 nulos — sem avaliação para tickets abertos | Mantidos como `null` |
| `tempo_resolucao_horas` | 2.072 nulos — tickets não resolvidos | Mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `ticket_id` |
| Datas | Formato ISO com timezone (`2023-01-03T05:13:00.000Z`) | Cast para `timestamp` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `resolvido` | `data_resolucao IS NOT NULL` |
| `hora_abertura` | `HOUR(data_abertura)` |
| `dia_semana_abertura` | `DAYOFWEEK(data_abertura)` |

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Usa a última ingestão de cada ticket (maior timestamp_ingestion) para
# garantir que reprocessamentos da Bronze não gerem duplicatas na Silver.

df_raw = spark.table(f'{bronze_schema}.suporte_tickets')

window_dedup = Window.partitionBy('ticket_id').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   # será recriado com o instante desta carga
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

Registros na Bronze  : 34,697
Após deduplicação    : 34,697
Duplicatas removidas : 0


In [0]:
# ─── Mapeamento canônico de tipo_problema ────────────────────────────────────
# Na Bronze foram encontradas 4 categorias reais fragmentadas em ~25 variações:
#
#   Entrega   - Entrega, ENTREGA, entrega, 3ntrega, entr, del, DELAY, delay
#   Reembolso - Reembolso, REEMBOLSO, reembolso, REFUND, ref, reemb, r3embolso
#   Produto   - Produto, PRODUTO, produto, pro, prod, p3oduto, PRODUCT, product
#   Pagamento - Pagamento, PAGAMENTO, pagamento, pag, pay, PAY, PAYMENT, payment,
#               p4gamento
#
# Estratégia: normaliza para lowercase e aplica mapeamento por prefixo/palavra-chave.
# Valores não mapeados são marcados como 'Outro' para investigação futura.

tipo_map = {
    # Entrega
    'entrega'  : 'Entrega',  '3ntrega' : 'Entrega',  'entr'  : 'Entrega',
    'del'      : 'Entrega',  'delay'   : 'Entrega',
    # Reembolso
    'reembolso': 'Reembolso','reemb'   : 'Reembolso','r3embolso': 'Reembolso',
    'refund'   : 'Reembolso','ref'     : 'Reembolso',
    # Produto
    'produto'  : 'Produto',  'pro'     : 'Produto',  'prod'  : 'Produto',
    'p3oduto'  : 'Produto',  'product' : 'Produto',
    # Pagamento
    'pagamento': 'Pagamento','pag'     : 'Pagamento','p4gamento': 'Pagamento',
    'pay'      : 'Pagamento','payment' : 'Pagamento',
}

# Constrói expressão CASE WHEN a partir do dicionário
tipo_expr = F.col('tipo_problema')
for raw_val, canonical in tipo_map.items():
    tipo_expr = F.when(
        F.lower(F.col('tipo_problema')) == raw_val, canonical
    ).otherwise(tipo_expr)

# Valores que não bateram com nenhum mapeamento ficam como 'Outro'
known_lower = [k.lower() for k in tipo_map.keys()]
tipo_expr = F.when(
    F.lower(F.col('tipo_problema')).isin(known_lower), tipo_expr
).otherwise(F.lit('Outro'))

print('Expressão de mapeamento criada.')

Expressão de mapeamento criada.


In [0]:
# ─── Transformações principais ───────────────────────────────────────────────

df_silver = (
    df_dedup

    # 1. Tipos corretos para datas (ISO 8601 com timezone)
    .withColumn('data_abertura',   F.to_timestamp('data_abertura'))
    .withColumn('data_resolucao',  F.to_timestamp('data_resolucao'))

    # 2. Normalização de tipo_problema
    .withColumn('tipo_problema', tipo_expr)

    # 3. Colunas derivadas de negócio
    .withColumn(
        'resolvido',
        F.col('data_resolucao').isNotNull()
    )
    .withColumn(
        'hora_abertura',
        F.hour('data_abertura').cast('int')
    )
    .withColumn(
        'dia_semana_abertura',
        F.date_format('data_abertura', 'EEEE')   # nome completo em inglês; adaptar locale se necessário
    )

    # 4. Tipos numéricos
    .withColumn('tempo_resolucao_horas', F.col('tempo_resolucao_horas').cast('decimal(10,2)'))
    .withColumn('nota_avaliacao',        F.col('nota_avaliacao').cast('decimal(3,1)'))

    # 5. Marca temporal desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação de colunas conforme schema Silver acordado
    .select(
        'ticket_id',
        'id_cliente',
        'id_pedido',
        'tipo_problema',
        'data_abertura',
        'data_resolucao',
        'tempo_resolucao_horas',
        'agente_suporte',
        'nota_avaliacao',
        'resolvido',
        'hora_abertura',
        'dia_semana_abertura',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

Transformações aplicadas. Schema resultante:
root
 |-- ticket_id: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_pedido: string (nullable = true)
 |-- tipo_problema: string (nullable = true)
 |-- data_abertura: timestamp (nullable = true)
 |-- data_resolucao: timestamp (nullable = true)
 |-- tempo_resolucao_horas: decimal(10,2) (nullable = true)
 |-- agente_suporte: string (nullable = true)
 |-- nota_avaliacao: decimal(3,1) (nullable = true)
 |-- resolvido: boolean (nullable = false)
 |-- hora_abertura: integer (nullable = true)
 |-- dia_semana_abertura: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)



In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total = df_silver.count()
resolvidos     = df_silver.filter(F.col('resolvido') == True).count()
nao_resolvidos = df_silver.filter(F.col('resolvido') == False).count()
outros_tipo    = df_silver.filter(F.col('tipo_problema') == 'Outro').count()

print(f'Total de tickets       : {total:,}')
print(f'Resolvidos             : {resolvidos:,} ({resolvidos/total*100:.1f}%)')
print(f'Abertos (sem resolução): {nao_resolvidos:,} ({nao_resolvidos/total*100:.1f}%)')
print(f'Tipo "Outro" (suspeitos): {outros_tipo:,}')
print()
print('Distribuição de tipo_problema após normalização:')
df_silver.groupBy('tipo_problema').count().orderBy(F.desc('count')).show()

Total de tickets       : 34,697
Resolvidos             : 32,625 (94.0%)
Abertos (sem resolução): 2,072 (6.0%)
Tipo "Outro" (suspeitos): 0

Distribuição de tipo_problema após normalização:
+-------------+-----+
|tipo_problema|count|
+-------------+-----+
|      Entrega|10517|
|    Reembolso| 8903|
|      Produto| 8574|
|    Pagamento| 6703|
+-------------+-----+



In [0]:
# ─── Grava silver.ft_tickets_suporte ─────────────────────────────────────────
# overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.ft_tickets_suporte')
)

print(f'{silver_schema}.ft_tickets_suporte gravada com {df_silver.count():,} registros.')

vcommerce_silver.ft_tickets_suporte gravada com 34,697 registros.


In [0]:
# ─── dim_tipos_problema ───────────────────────────────────────────────────────
# Dimensão com os 4 tipos canônicos e sua categoria de negócio.
#
# Categoria derivada:
#   Entrega   - Logística
#   Reembolso - Financeiro
#   Produto   - Qualidade
#   Pagamento - Financeiro
#   Outro     - Indefinido

df_dim_tipos = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .select('tipo_problema')
         .distinct()
         .withColumn(
             'categoria_problema',
             F.when(F.col('tipo_problema') == 'Entrega',   'Logística')
              .when(F.col('tipo_problema') == 'Reembolso', 'Financeiro')
              .when(F.col('tipo_problema') == 'Produto',   'Qualidade')
              .when(F.col('tipo_problema') == 'Pagamento', 'Financeiro')
              .otherwise('Indefinido')
         )
         .orderBy('tipo_problema')
)

df_dim_tipos.show()

(
    df_dim_tipos.write
                .format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(f'{silver_schema}.dim_tipos_problema')
)

print(f'{silver_schema}.dim_tipos_problema gravada.')

+-------------+------------------+
|tipo_problema|categoria_problema|
+-------------+------------------+
|      Entrega|         Logística|
|    Pagamento|        Financeiro|
|      Produto|         Qualidade|
|    Reembolso|        Financeiro|
+-------------+------------------+

vcommerce_silver.dim_tipos_problema gravada.


In [0]:
# ─── dim_agentes_suporte ──────────────────────────────────────────────────────
# Dimensão com métricas agregadas por agente, derivadas de ft_tickets_suporte.
#
# Métricas calculadas:
#   qtd_tickets_resolvidos  - total de tickets onde resolvido = true
#   nota_media_atendimento  - média de nota_avaliacao (ignora nulos)

df_dim_agentes = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .groupBy('agente_suporte')
         .agg(
             F.count(
                 F.when(F.col('resolvido') == True, 1)
             ).alias('qtd_tickets_resolvidos'),
             F.round(
                 F.avg('nota_avaliacao'), 2
             ).alias('nota_media_atendimento'),
         )
         .withColumn(
             'nota_media_atendimento',
             F.col('nota_media_atendimento').cast('decimal(4,2)')
         )
         .orderBy(F.desc('qtd_tickets_resolvidos'))
)

df_dim_agentes.show(truncate=False)

(
    df_dim_agentes.write
                  .format('delta')
                  .mode('overwrite')
                  .option('overwriteSchema', 'true')
                  .saveAsTable(f'{silver_schema}.dim_agentes_suporte')
)

print(f'{silver_schema}.dim_agentes_suporte gravada.')

+--------------------+----------------------+----------------------+
|agente_suporte      |qtd_tickets_resolvidos|nota_media_atendimento|
+--------------------+----------------------+----------------------+
|Carlos Eduardo Silva|5793                  |3.76                  |
|Ana Paula Ferreira  |5589                  |3.80                  |
|Mariana Costa       |4924                  |3.71                  |
|Fernanda Lima       |2325                  |3.26                  |
|Bruno Rodrigues     |2273                  |3.36                  |
|Rafael Oliveira     |2268                  |3.54                  |
|Juliana Santos      |2250                  |3.39                  |
|Thiago Alves        |1963                  |3.17                  |
|Camila Nascimento   |820                   |3.11                  |
|Lucas Pereira       |753                   |3.03                  |
|Eduardo Cardoso     |389                   |3.17                  |
|Felipe Araújo       |380         

---

## Tratamento: `ft_clickstream`

**Origem:** `bronze.clickstream`  
**Destino:** `silver.ft_clickstream`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `tipo_evento` | Múltiplas variações para mesmas categorias (`view`, `VISUALIZACAO`, `page_view`…) | Mapeamento para 6 valores canônicos |
| `dispositivo` | Inconsistências de case e sinônimos (`MOBILE`, `celular`, `smartphone`) | Normalização para `Desktop`, `Mobile`, `Tablet` |
| `origem_sessao` | Variações em inglês/português e case (`organic`, `ORGANICO`, `cpc`…) | Normalização para 6 canais canônicos |
| `timestamp_evento` | Formato ISO 8601 com timezone lido como `string` na Bronze | Cast para `timestamp` |
| `tempo_pagina_seg` | Possíveis valores negativos ou zero (sessões inválidas) | Substituídos por `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze | Deduplicação por `id_evento` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `hora_evento` | `HOUR(timestamp_evento)` |
| `dia_semana_evento` | `date_format(timestamp_evento, 'EEEE')` |
| `periodo_dia` | Madrugada (0–5 h), Manhã (6–11 h), Tarde (12–17 h), Noite (18–23 h) |
| `is_conversao` | `tipo_evento == 'Compra'` |
| `ano_mes_evento` | `date_format(timestamp_evento, 'yyyy-MM')` para agregações mensais |

In [0]:
# ─── Leitura da Bronze e deduplicação ───────────────────────────────────────
# event_id é a chave primária de cada evento de navegação.
# Mantém apenas o registro mais recente de cada event_id para neutralizar
# reprocessamentos em modo append na Bronze.

df_raw_cs = spark.table(f'{bronze_schema}.clickstream')

window_dedup_cs = Window.partitionBy('id_evento').orderBy(F.desc('timestamp_ingestion'))

df_dedup_cs = (
    df_raw_cs
    .withColumn('_rank', F.row_number().over(window_dedup_cs))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')
)

total_raw_cs   = df_raw_cs.count()
total_dedup_cs = df_dedup_cs.count()
print(f'Registros na Bronze  : {total_raw_cs:,}')
print(f'Após deduplicação    : {total_dedup_cs:,}')
print(f'Duplicatas removidas : {total_raw_cs - total_dedup_cs:,}')

Registros na Bronze  : 500,000
Após deduplicação    : 500,000
Duplicatas removidas : 0


In [0]:
# ─── Mapeamento canônico de tipo_evento ────────────────────────────────────
# 6 categorias reais encontradas com múltiplas grafias na Bronze:
#
#   Visualizacao - visualizacao, VISUALIZACAO, view, VIEW, page_view, pageview
#   Clique       - clique, CLIQUE, click, CLICK
#   Carrinho     - add_carrinho, ADD_CARRINHO, add_to_cart, addtocart, adicionar_carrinho
#   Checkout     - checkout, CHECKOUT, chk
#   Compra       - compra, COMPRA, purchase, PURCHASE, buy
#   Busca        - busca, BUSCA, search, SEARCH

tipo_evento_map = {
    # Visualização
    'visualizacao' : 'Visualização',
    'view'         : 'Visualização',
    'page_view'    : 'Visualização',
    'pageview'     : 'Visualização',
    'page-view'    : 'Visualização',
    'pv'           : 'Visualização',
    'product_view' : 'Visualização',
    # Clique
    'clique' : 'Clique',
    'click'  : 'Clique',
    # Carrinho
    'add_carrinho'       : 'Carrinho',
    'add_to_cart'        : 'Carrinho',
    'addtocart'          : 'Carrinho',
    'adicionar_carrinho' : 'Carrinho',
    'adicionar'          : 'Carrinho',
    # Checkout
    'checkout' : 'Checkout',
    'chk'      : 'Checkout',
    # Compra
    'compra'   : 'Compra',
    'purchase' : 'Compra',
    'buy'      : 'Compra',
    # Busca
    'busca'  : 'Busca',
    'search' : 'Busca',
    'srch'   : 'Busca',
    # Outros eventos relevantes
    'login'         : 'Login',
    'abandon_cart'  : 'Abandono Carrinho',
}

tipo_evento_expr = F.col('tipo_evento')
for raw_val, canonical in tipo_evento_map.items():
    tipo_evento_expr = F.when(
        F.lower(F.col('tipo_evento')) == raw_val, canonical
    ).otherwise(tipo_evento_expr)

known_tipos = list(tipo_evento_map.keys())
tipo_evento_expr = F.when(
    F.lower(F.col('tipo_evento')).isin(known_tipos), tipo_evento_expr
).otherwise(F.lit('Outro'))

print('Expressão tipo_evento criada.')

Expressão tipo_evento criada.


In [0]:
# ─── Mapeamento canônico de dispositivo e canal ────────────────────────────

dispositivo_map = {
    # Desktop
    'desktop'    : 'Desktop',
    'computador' : 'Desktop',
    # Mobile
    'mobile'     : 'Mobile',
    'smartphone' : 'Mobile',
    'celular'    : 'Mobile',
    'mob'        : 'Mobile',
    # Tablet
    'tablet' : 'Tablet',
    'tab'    : 'Tablet',
}

dispositivo_expr = F.col('dispositivo')
for raw_val, canonical in dispositivo_map.items():
    dispositivo_expr = F.when(
        F.lower(F.col('dispositivo')) == raw_val, canonical
    ).otherwise(dispositivo_expr)

known_disp = list(dispositivo_map.keys())
dispositivo_expr = F.when(
    F.lower(F.col('dispositivo')).isin(known_disp), dispositivo_expr
).otherwise(F.lit('Outro'))

# ── Canal de tráfego ──────────────────────────────────────────────────────────
origem_sessao_map = {
    'organico'      : 'Organico',
    'organic'       : 'Organico',
    'pago'          : 'Pago',
    'paid'          : 'Pago',
    'paid_search'   : 'Pago',
    'ads'           : 'Pago',
    'social'        : 'Redes Sociais',
    'redes_sociais' : 'Redes Sociais',
    'social_media'  : 'Redes Sociais',
    'email'         : 'Email',
    'e-mail'        : 'Email',
    'direto'        : 'Direto',
    'direct'        : 'Direto',
}

origem_sessao_expr = F.col('origem_sessao')
for raw_val, canonical in origem_sessao_map.items():
    origem_sessao_expr = F.when(
        F.lower(F.col('origem_sessao')) == raw_val, canonical
    ).otherwise(origem_sessao_expr)

known_origin = list(origem_sessao_map.keys())
origem_sessao_expr = F.when(
    F.lower(F.col('origem_sessao')).isin(known_origin), origem_sessao_expr
).otherwise(F.lit('Outro'))

print('Expressões dispositivo e origem_sessao criadas.')

Expressões dispositivo e origem_sessao criadas.


In [0]:
# ── Canal de marketing (agrupamento de nível superior) ───────────────────────
canal_map = {
  # Web
  'web' : 'Web',
  # App
  'app'        : 'App',
  'aplicativo' : 'App',
  # Mobile Web
  'mobile_web' : 'Mobile Web',
  'mobile web' : 'Mobile Web',
  # Extras (caso apareçam no futuro)
  'mobile'  : 'Mobile Web',
  'desktop' : 'Web',
}

canal_expr = F.col('canal')
for raw_val, canonical in canal_map.items():
    canal_expr = F.when(
        F.lower(F.col('canal')) == raw_val, canonical
    ).otherwise(canal_expr)

known_canal = list(canal_map.keys())
canal_expr = F.when(
    F.lower(F.col('canal')).isin(known_canal), canal_expr
).otherwise(F.lit('Outro'))

print('Expressoes canal criadas.')

Expressoes canal criadas.


In [0]:
# ─── Transformações principais ───────────────────────────────────────────────

df_silver_cs = (
    df_dedup_cs

    # 1. Tipo correto para o timestamp do evento
    .withColumn('data_evento', F.to_timestamp('data_evento'))

    # 2. Normalização de categóricas
    .withColumn('tipo_evento', tipo_evento_expr)
    .withColumn('dispositivo', dispositivo_expr)
    .withColumn('origem_sessao',origem_sessao_expr)
    .withColumn('canal', canal_expr)

    # 3. Tipos numéricos
    .withColumn('tempo_pagina_seg', F.col('tempo_pagina_seg').cast('int'))

    # 4. Sessões inválidas: duração negativa ou zero substituída por null
    .withColumn(
        'tempo_pagina_seg',
        F.when(F.col('tempo_pagina_seg') > 0, F.col('tempo_pagina_seg')).otherwise(F.lit(None))
    )

    # 5. Colunas derivadas de tempo
    .withColumn('hora_evento',       F.hour('data_evento').cast('int'))
    .withColumn('dia_semana_evento', F.date_format('data_evento', 'EEEE'))
    .withColumn('ano_mes_evento',    F.date_format('data_evento', 'yyyy-MM'))

    # 6. Período do dia baseado na hora
    .withColumn(
        'periodo_dia',
        F.when(F.col('hora_evento').between(0,  5), 'Madrugada')
         .when(F.col('hora_evento').between(6, 11), 'Manha')
         .when(F.col('hora_evento').between(12, 17), 'Tarde')
         .otherwise('Noite')
    )

    # 7. Flag de conversão (evento que representa compra concluída)
    .withColumn(
        'is_conversao',
        F.col('tipo_evento') == 'Compra'
    )

    # 8. Marca temporal desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 9. Ordenação de colunas do schema Silver
    .select(
        'id_evento',
        'id_sessao',
        'id_cliente',
        'id_dispositivo',
        'id_produto',
        'tipo_evento',
        'canal',
        'dispositivo',
        'origem_sessao',
        'data_evento',
        'hora_evento',
        'dia_semana_evento',
        'periodo_dia',
        'ano_mes_evento',
        'tempo_pagina_seg',
        'is_conversao',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver_cs.printSchema()

Transformações aplicadas. Schema resultante:
root
 |-- id_evento: string (nullable = true)
 |-- id_sessao: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_dispositivo: string (nullable = true)
 |-- id_produto: string (nullable = true)
 |-- tipo_evento: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- dispositivo: string (nullable = true)
 |-- origem_sessao: string (nullable = true)
 |-- data_evento: timestamp (nullable = true)
 |-- hora_evento: integer (nullable = true)
 |-- dia_semana_evento: string (nullable = true)
 |-- periodo_dia: string (nullable = false)
 |-- ano_mes_evento: string (nullable = true)
 |-- tempo_pagina_seg: integer (nullable = true)
 |-- is_conversao: boolean (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)



In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total_cs    = df_silver_cs.count()
conversoes  = df_silver_cs.filter(F.col('is_conversao') == True).count()
sem_cliente = df_silver_cs.filter(F.col('id_cliente').isNull()).count()
sem_duracao = df_silver_cs.filter(F.col('tempo_pagina_seg').isNull()).count()
outros_tipo = df_silver_cs.filter(F.col('tipo_evento') == 'Outro').count()

print(f'Total de eventos           : {total_cs:,}')
print(f'Conversoes (Compra)        : {conversoes:,} ({conversoes/total_cs*100:.1f}%)')
print(f'Sessoes sem duracao valida : {sem_duracao:,} ({sem_duracao/total_cs*100:.1f}%)')
print(f'Clientes anonimos (null)   : {sem_cliente:,} ({sem_cliente/total_cs*100:.1f}%)')
print(f'tipo_evento "Outro"        : {outros_tipo:,}')
print()
print('Distribuicao de tipo_evento apos normalizacao:')
df_silver_cs.groupBy('tipo_evento').count().orderBy(F.desc('count')).show()
print('Distribuicao de dispositivo:')
df_silver_cs.groupBy('dispositivo').count().orderBy(F.desc('count')).show()
print('Distribuicao de origem_sessao:')
df_silver_cs.groupBy('origem_sessao').count().orderBy(F.desc('count')).show()
print('Distribuicao de canal:')
df_silver_cs.groupBy('canal').count().orderBy(F.desc('count')).show()
print('Distribuicao de periodo_dia:')
df_silver_cs.groupBy('periodo_dia').count().orderBy(F.desc('count')).show()

Total de eventos           : 500,000
Conversoes (Compra)        : 5,437 (1.1%)
Sessoes sem duracao valida : 44,977 (9.0%)
Clientes anonimos (null)   : 150,553 (30.1%)
tipo_evento "Outro"        : 18,692

Distribuicao de tipo_evento apos normalizacao:
+-----------------+------+
|      tipo_evento| count|
+-----------------+------+
|     Visualização|290922|
|            Busca| 74946|
|         Carrinho| 62942|
|            Login| 30472|
|            Outro| 18692|
|         Checkout| 13318|
|           Compra|  5437|
|Abandono Carrinho|  3271|
+-----------------+------+

Distribuicao de dispositivo:
+-----------+------+
|dispositivo| count|
+-----------+------+
|    Desktop|238079|
|     Mobile|211909|
|     Tablet| 50012|
+-----------+------+

Distribuicao de origem_sessao:
+-------------+------+
|origem_sessao| count|
+-------------+------+
|     Organico|199812|
|         Pago|125145|
|       Direto|100012|
|Redes Sociais| 49830|
|        Email| 25201|
+-------------+------+

Distribu

In [0]:
# ─── Grava silver.ft_clickstream ────────────────────────────────────────────
# overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver_cs.write
                .format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(f'{silver_schema}.ft_clickstream')
)

print(f'{silver_schema}.ft_clickstream gravada com {df_silver_cs.count():,} registros.')

vcommerce_silver.ft_clickstream gravada com 500,000 registros.


In [0]:
# ─── Resumo final ────────────────────────────────────────────────────────────

tabelas = [
    f'{silver_schema}.ft_tickets_suporte',
    f'{silver_schema}.dim_tipos_problema',
    f'{silver_schema}.dim_agentes_suporte',
    f'{silver_schema}.ft_clickstream',
]

print('=== Camada Silver — Resumo ===')
print(f'{"Tabela":<40} {"Linhas":>8} {"Colunas":>8}')
print('-' * 60)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<40} {df_t.count():>8,} {len(df_t.columns):>8}')

=== Camada Silver — Resumo ===
Tabela                                     Linhas  Colunas
------------------------------------------------------------
vcommerce_silver.ft_tickets_suporte        34,697       13
vcommerce_silver.dim_tipos_problema             4        2
vcommerce_silver.dim_agentes_suporte           20        3
vcommerce_silver.ft_clickstream           500,000       17


---

## Tratamento: `ft_avaliacoes`

**Origem:** `bronze.avaliacoes`  
**Destino:** `silver.ft_avaliacoes`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `nota_produto` | Valores fora do range 1–5 (`-1`, `0`, `6`) e textos (`bom`, `ruim`, `péssimo`, `ótimo`) | Mapear textos para numérico; anular valores fora de 1–5 com `null` |
| `nota_nps` | Valores fora do range 0–10 (`-1`, `11`) e mesmos textos acima | Mapear textos para numérico; anular valores fora de 0–10 com `null` |
| `recomenda` | ~12 variações (`S`, `sim`, `SIM`, `yes`, `1` / `N`, `Nao`, `NAO`, `no`, `0`) | Normalizar para booleano (`true` / `false`) |
| `data_avaliacao` | 3.174 registros nulos; formato `datetime com espaço` | Cast para `timestamp`; nulos mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `id_avaliacao` |

### Colunas derivadas criadas

| Coluna | Lógica |
|---|---|
| `categoria_nps` | Detrator (0–6) · Neutro (7–8) · Promotor (9–10) — `null` quando `nota_nps` é inválida |
| `recomenda` | Convertido de texto/número para boolean |


In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Lê todos os registros brutos da tabela de avaliações ingerida na camada Bronze.
# Como a Bronze usa mode=append, reprocessamentos podem gerar registros duplicados
# para o mesmo id_avaliacao. Usamos row_number() sobre a janela particionada por
# id_avaliacao (ordenada pelo timestamp_ingestion mais recente) para ficar apenas
# com a versão mais atual de cada avaliação.

df_raw = spark.table(f'{bronze_schema}.avaliacoes')

window_dedup = Window.partitionBy('id_avaliacao').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')
)

total_raw = df_raw.count()
total_dedup = df_dedup.count()

print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')


Registros na Bronze  : 156,832
Após deduplicação    : 156,832
Duplicatas removidas : 0


In [0]:
# ─── Mapeamento de notas textuais -> numéricas ────────────────────────────────
# Na Bronze, tanto nota_produto quanto nota_nps apresentam valores textuais
# misturados com valores numéricos. Os textos encontrados e seus equivalentes
# numéricos adotados são:
#
#   'ótimo'   -> 5   (nota máxima)
#   'bom'     -> 4   (nota boa)
#   'ruim'    -> 2   (nota ruim)
#   'péssimo' -> 1   (nota mínima)
#
# Após a conversão textual, valores fora dos ranges válidos são anulados:
#   nota_produto : range esperado 1–5  → fora disso vira null
#   nota_nps     : range esperado 0–10 → fora disso vira null

texto_para_nota = {
    'ótimo':   5,
    'bom':     4,
    'ruim':    2,
    'péssimo': 1,
}
# Constrói expressão que converte texto para número antes do cast final.
# Para cada par (texto, valor) no dicionário, encadeia um WHEN; o OTHERWISE
# mantém o valor original (caso já seja numérico em string).

def texto_para_numero_da_nota(coluna: str):
    # Essa função retorna uma coluna Pyspark que normaliza os textos para numero int
    expressao = F.col(coluna)
    for texto, nota in texto_para_nota.items():
        # Cast final para int e valores que nao sao numericos que sobrarem viram null
        expressao = F.when(F.lower(F.col(coluna)) == texto, F.lit(nota)).otherwise(expressao)
    return expressao.cast('int')

print('Expressões de mapeamento criadas.')
print(f'Textos mapeados: {list(texto_para_nota.keys())}')

Expressões de mapeamento criadas.
Textos mapeados: ['ótimo', 'bom', 'ruim', 'péssimo']


In [0]:
# ─── Mapeamento de recomenda -> boolean ───────────────────────────────────────
# O campo recomenda chegou com ~12 variações para dois valores semânticos:
#
#   Positivo (true)  : 'S', 'Sim', 'SIM', 'sim', 'yes', '1'
#   Negativo (false) : 'N', 'Nao', 'NAO', 'nao', 'no',  '0'
#
# A estratégia é normalizar para lowercase e checar pertencimento ao conjunto
# positivo. Qualquer valor não reconhecido vira null para não inferir intenção.

valor_recomenda_positivo = {'s', 'sim', 'yes', '1'}
valor_recomenda_negativo = {'n', 'nao', 'no', '0'}

expressao_recomenda = (
    F.when(F.lower(F.col('recomenda')).isin(valor_recomenda_positivo), True)
    .when(F.lower(F.col('recomenda')).isin(valor_recomenda_negativo), False)
    .otherwise(None) #aqui é para os valores que a gente nao conhece ai não inferimos
    .cast('boolean')
)

print('Expressão de normalização de recomenda criada.')
print(f'  Positivo (true) : {valor_recomenda_positivo}')
print(f'  Negativo (false): {valor_recomenda_negativo}')

Expressão de normalização de recomenda criada.
  Positivo (true) : {'sim', 'yes', '1', 's'}
  Negativo (false): {'0', 'n', 'no', 'nao'}


In [0]:
# ─── Transformações principais ───────────────────────────────────────────────
# Aplica sequencialmente todos os tratamentos definidos acima sobre o DataFrame
# deduplicado, produzindo o DataFrame Silver final.

df_silver = (
    df_dedup

    # 1. Converte notas textuais para inteiro e anula valores fora do range válido
    #    nota_produto: range 1–5 (escala de satisfação do produto)
    #    nota_nps: range 0–10 (Net Promoter Score)
    .withColumn('nota_produto',
        F.when(
            texto_para_numero_da_nota('nota_produto').between(1, 5),
            texto_para_numero_da_nota('nota_produto')
        ).otherwise(F.lit(None).cast('int'))   # fora do range -> null
    )
    .withColumn('nota_nps',
        F.when(
            texto_para_numero_da_nota('nota_nps').between(0, 10),
            texto_para_numero_da_nota('nota_nps')
        ).otherwise(F.lit(None).cast('int'))   # fora do range -> null
    )

    # 2. Normaliza recomenda para boolean usando a expressão construída acima
    .withColumn('recomenda', expressao_recomenda)

     # 3. Cast de data_avaliacao para timestamp
    #    Dois formatos coexistem na Bronze:
    #      - Padrão  : 'yyyy-MM-dd HH:mm:ss'  -> maioria dos registros
    #      - Variante: 'dd/MM/yyyy HH:mm:ss'  -> ~12.5k registros com barras
    #    try_to_timestamp é usado no lugar de to_timestamp pois nunca lança
    #    exceção — retorna null quando o valor não bate com o formato, o que
    #    torna o coalesce seguro mesmo com dados malformados.
    .withColumn('data_avaliacao', 
        F.coalesce(
            # Tenta o formato padrão com hífen (Ano-Mês-Dia)
            F.expr("try_to_timestamp(data_avaliacao, 'yyyy-MM-dd HH:mm:ss')"),    
            # Tenta o formato com barras (Ano/Dia/Mês) - O primeiro que deu erro
            F.expr("try_to_timestamp(data_avaliacao, 'yyyy/dd/MM HH:mm:ss')"),
            # Tenta o formato Brasileiro com barras (Dia/Mês/Ano) - Visto no print
            F.expr("try_to_timestamp(data_avaliacao, 'dd/MM/yyyy HH:mm:ss')"), 
            # Tenta o formato Americano com hifens (Mês-Dia-Ano) - Visto no print
            F.expr("try_to_timestamp(data_avaliacao, 'MM-dd-yyyy HH:mm:ss')"),
            # Fallback nativo do Spark para tentar salvar o que sobrar
            F.expr("try_to_timestamp(data_avaliacao)")
        )
    )

    # 4. Categoria NPS derivada — classificação padrão de mercado:
    #      0–6  -> Detrator  (clientes insatisfeitos, risco de churn)
    #      7–8  -> Neutro    (clientes passivos, sem engajamento forte)
    #      9–10 -> Promotor  (clientes leais, propensos a indicar)
    #    Quando nota_nps é null (valor inválido na origem), categoria_nps
    #    também fica null para não distorcer análises de NPS.
    .withColumn('categoria_nps',
        F.when(F.col('nota_nps').between(0, 6),  'Detrator')
         .when(F.col('nota_nps').between(7, 8),  'Neutro')
         .when(F.col('nota_nps').between(9, 10), 'Promotor')
         .otherwise(None)
    )

    # 5. Marca temporal desta carga Silver (recriado para rastreabilidade)
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação de colunas conforme schema Silver acordado
    .select(
        'id_avaliacao',
        'id_pedido',
        'id_cliente',
        'id_produto',
        'nota_produto',
        'comentario',
        'nota_nps',
        'categoria_nps',
        'recomenda',
        'data_avaliacao',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

Transformações aplicadas. Schema resultante:
root
 |-- id_avaliacao: string (nullable = true)
 |-- id_pedido: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_produto: string (nullable = true)
 |-- nota_produto: integer (nullable = true)
 |-- comentario: string (nullable = true)
 |-- nota_nps: integer (nullable = true)
 |-- categoria_nps: string (nullable = true)
 |-- recomenda: boolean (nullable = true)
 |-- data_avaliacao: timestamp (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)



In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────
# Antes de gravar, verificamos as principais métricas de qualidade para garantir
# que os tratamentos produziram o resultado esperado. Qualquer número suspeito
# deve ser investigado antes de prosseguir.

total = df_silver.count()

# Notas que viraram null após tratamento (eram inválidas na origem)
notas_produto_nulas = df_silver.filter(F.col('nota_produto').isNull()).count()
notas_nps_nulas = df_silver.filter(F.col('nota_nps').isNull()).count()
recomenda_nulas = df_silver.filter(F.col('recomenda').isNull()).count()
datas_nulas = df_silver.filter(F.col('data_avaliacao').isNull()).count()

print(f'Total de avaliações: {total:,}')
print()
print(f'nota_produto nulas (inválidas): {notas_produto_nulas:,} ({notas_produto_nulas/total*100:.1f}%)')
print(f'nota_nps nulas (inválidas): {notas_nps_nulas:,} ({notas_nps_nulas/total*100:.1f}%)')
print(f'recomenda nulas: {recomenda_nulas:,} ({recomenda_nulas/total*100:.1f}%)')
print(f'data_avaliacao nulas: {datas_nulas:,} ({datas_nulas/total*100:.1f}%)')
print()
print('Distribuição de categoria_nps:')
df_silver.groupBy('categoria_nps').count().orderBy('categoria_nps').show()
print('Distribuição de recomenda:')
df_silver.groupBy('recomenda').count().orderBy('recomenda').show()
print('Distribuição de nota_produto (range 1–5):')
df_silver.groupBy('nota_produto').count().orderBy('nota_produto').show()

Total de avaliações: 156,832

nota_produto nulas (inválidas): 1,415 (0.9%)
nota_nps nulas (inválidas): 1,075 (0.7%)
recomenda nulas: 0 (0.0%)
data_avaliacao nulas: 3,174 (2.0%)

Distribuição de categoria_nps:
+-------------+-----+
|categoria_nps|count|
+-------------+-----+
|         NULL| 1075|
|     Detrator|40842|
|       Neutro|50879|
|     Promotor|64036|
+-------------+-----+

Distribuição de recomenda:
+---------+------+
|recomenda| count|
+---------+------+
|    false| 39580|
|     true|117252|
+---------+------+

Distribuição de nota_produto (range 1–5):
+------------+-----+
|nota_produto|count|
+------------+-----+
|        NULL| 1415|
|           1| 9449|
|           2|16574|
|           3|30038|
|           4|60951|
|           5|38405|
+------------+-----+



In [0]:
# ─── Grava silver.ft_avaliacoes ──────────────────────────────────────────────
# Usa mode=overwrite para garantir idempotência: reexecutar este notebook
# sempre produz o mesmo resultado, sem acumular registros duplicados.
# overwriteSchema=true permite que alterações de schema futuras não quebrem
# a execução.

df_silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{silver_schema}.ft_avaliacoes')

print(f'{silver_schema}.ft_avaliacoes gravadas com {df_silver.count():,} registros.')

vcommerce_silver.ft_avaliacoes gravadas com 156,832 registros.


In [0]:
# ─── Resumo final ─────────────────────────────────────────────────────────────
# Confirma a tabela gravada e exibe contagem e número de colunas,
# seguindo o mesmo padrão de log das outras seções deste notebook.

tabelas = [
    f'{silver_schema}.ft_avaliacoes',
]

print('=== Camada Silver — Avaliações Pós-Compra ===')
print(f'{"Tabela":<40} {"Linhas":>8} {"Colunas":>8}')
print('-' * 60)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<40} {df_t.count():>8,} {len(df_t.columns):>8}')

=== Camada Silver — Avaliações Pós-Compra ===
Tabela                                     Linhas  Colunas
------------------------------------------------------------
vcommerce_silver.ft_avaliacoes            156,832       11


# Tratamento Silver — `dim_categorias_produto`

---

**Origem:** `bronze.catalogo_produtos`  
**Destino:** `silver.dim_categorias_produto`

---

## Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `categoria` | **Aproximadamente 50 variações** para 8 categorias reais, distribuídas em: Variações de case (`VESTUARIO`, `eletronicos`), tipos com substituição de letras por números (`M0VEIS`, `Cas4`, `3sportes`, `automotiv3`, `B3LEZA`, `BR1NQUEDOS`), abreviações (`vest`, `elet`, `brin`, `cas`, `esp`, `aut`, `mov`, `bel`), acentuação inconsistente (`Eletrônico`, `Móveis`) e **27 nulos** | Normalização para lowercase + Mapeamento para 8 valores canônicos + Nulos descartados |

---

## Mapeamento canônico de `categoria`

As 8 categorias reais identificadas e todas as variações encontradas no dataset:

| Valor Canônico | Variações mapeadas |
|---|---|
| `Eletronicos` | `eletronicos`, `ELETRONICOS`, `Eletrônico`, `Eletronico`, `elet`, `ELET` |
| `Vestuario` | `VESTUARIO`, `vestuario`, `Vestuarios`, `vest`, `vestu`, `VEST` |
| `Moveis` | `moveis`, `MOVEIS`, `M0VEIS`, `Móveis`, `mov` |
| `Casa` | `casa`, `CASA`, `cas`, `cas@`, `Cas4` |
| `Esportes` | `esportes`, `ESPORTES`, `Esporte`, `esport`, `ESPORT`, `esp`, `3sportes` |
| `Beleza` | `beleza`, `BELEZA`, `B3LEZA`, `bel`, `Belz` |
| `Brinquedos` | `brinquedos`, `Brinquedo`, `BR1NQUEDOS`, `brin`, `Brinq` |
| `Automotivo` | `automotivo`, `Automotivo`, `automotiv3`, `aut`, `Autom` |

---

## Colunas derivadas criadas

| Coluna | Lógica |
|---|---|
| `segmento` | Agrupamento de negócio derivado da categoria canônica |

### Mapeamento de categorias para segmentos

| Categoria | Segmento |
|---|---|
| `Eletronicos` | Tecnologia |
| `Vestuario` | Moda |
| `Moveis` | Casa & Decoração |
| `Casa` | Casa & Decoração |
| `Esportes` | Bem-Estar |
| `Beleza` | Bem-Estar |
| `Brinquedos` | Entretenimento |
| `Automotivo` | Automotivo |

---

## Schema final — `silver.dim_categorias_produto`

| Coluna | Tipo | Observação |
|---|---|---|
| `categoria` | `string` | Valor canônico — chave desta dimensão; 1 registro por categoria |
| `segmento` | `string` | Agrupamento de negócio derivado |
| `timestamp_ingestion` | `timestamp` | Instante da carga Silver |

---

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Usa a última ingestão de cada produto (maior timestamp_ingestion) para
# garantir que reprocessamentos da Bronze não gerem duplicatas na Silver.

df_raw = spark.table(f'{bronze_schema}.catalogo_produtos')

window_dedup = Window.partitionBy('id_produto').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   # será recriado com o instante desta carga
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

Registros na Bronze  : 517
Após deduplicação    : 517
Duplicatas removidas : 0


In [0]:
# ─── Mapeamento canônico de categoria ────────────────────────────────────────
# Na Bronze foram encontradas 8 categorias reais fragmentadas em aproximadamente 50 variações:
#
#   Eletronicos -> eletronicos, ELETRONICOS, Eletrônico, Eletronico, elet, ELET
#   Vestuario   -> VESTUARIO, vestuario, Vestuarios, vest, vestu, VEST
#   Moveis      -> moveis, MOVEIS, M0VEIS, Móveis, mov
#   Casa        -> casa, CASA, cas, cas@, Cas4
#   Esportes    -> esportes, ESPORTES, Esporte, esport, ESPORT, esp, 3sportes
#   Beleza      -> beleza, BELEZA, B3LEZA, bel, Belz
#   Brinquedos  -> brinquedos, Brinquedo, BR1NQUEDOS, brin, Brinq
#   Automotivo  -> automotivo, Automotivo, automotiv3, aut, Autom
#
# Estratégia: Normaliza-se os valores para lowercase e aplicamos mapeamento por valor exato.
# Observação: Valores não reconhecidos e nulos são descartados — por se tratar de uma
# dimensão de referência, um valor desconhecido não deve entrar no catálogo canônico.

# Mapeamento das categorias:
categoria_map = {
    # Eletronicos
    'eletronicos' : 'Eletronicos', 'eletrônicos' : 'Eletronicos',
    'eletronico'  : 'Eletronicos', 'eletrônico'  : 'Eletronicos',
    'elet'        : 'Eletronicos', 'ELET'         : 'Eletronicos',
    # Vestuario
    'vestuario'   : 'Vestuario',   'vestuários'  : 'Vestuario',
    'vestuarios'  : 'Vestuario',   'vest'         : 'Vestuario',
    'vestu'       : 'Vestuario',
    # Moveis
    'moveis'      : 'Moveis',      'móveis'       : 'Moveis',
    'm0veis'      : 'Moveis',      'mov'           : 'Moveis',
    # Casa
    'casa'        : 'Casa',        'cas@'          : 'Casa',
    'cas4'        : 'Casa',        'cas'           : 'Casa',
    # Esportes
    'esportes'    : 'Esportes',    'esporte'       : 'Esportes',
    'esport'      : 'Esportes',    'esp'           : 'Esportes',
    '3sportes'    : 'Esportes',
    # Beleza
    'beleza'      : 'Beleza',      'b3leza'        : 'Beleza',
    'bel'         : 'Beleza',      'belz'          : 'Beleza',
    # Brinquedos
    'brinquedos'  : 'Brinquedos',  'brinquedo'    : 'Brinquedos',
    'br1nquedos'  : 'Brinquedos',  'brin'          : 'Brinquedos',
    'brinq'       : 'Brinquedos',
    # Automotivo
    'automotivo'  : 'Automotivo',  'automotiv3'   : 'Automotivo',
    'aut'         : 'Automotivo',  'autom'         : 'Automotivo',
}

# Constrói a expressão CASE WHEN a partir do dicionário.
# Normaliza para lowercase antes de comparar para cobrir variações de case.
categoria_expr = F.lit(None).cast('string')
for raw_val, canonical in categoria_map.items():
    categoria_expr = F.when(
        F.lower(F.trim(F.col('categoria'))) == raw_val.lower(), canonical
    ).otherwise(categoria_expr)

print(f'Expressão de mapeamento criada para {len(categoria_map)} variações.')

Expressão de mapeamento criada para 37 variações.


In [0]:
# ─── Transformações principais ───────────────────────────────────────────────
# Aplica-se o mapeamento canônico, descarta nulos e valores não reconhecidos.
# Deduplica para 1 registro por categoria e deriva o segmento de negócio.
#
# Segmentos derivados:
#   Eletronicos -> Tecnologia
#   Vestuario   -> Moda
#   Moveis      -> Casa & Decoração
#   Casa        -> Casa & Decoração
#   Esportes    -> Bem-Estar
#   Beleza      -> Bem-Estar
#   Brinquedos  -> Entretenimento
#   Automotivo  -> Automotivo

# Transformação principal:
df_silver = (
    df_dedup

    # 1. Aplica mapeamento canônico — valores não reconhecidos e nulos ficam 'null'
    .withColumn('categoria', categoria_expr)

    # 2. Descarta registros sem categoria canônica (nulos originais + não mapeados)
    .filter(F.col('categoria').isNotNull())

    # 3. Mantém apenas a coluna categoria e deduplica para 1 linha por categoria
    .select('categoria')
    .distinct()

    # 4. Deriva segmento de negócio a partir da categoria canônica
    .withColumn(
        'segmento',
        F.when(F.col('categoria') == 'Eletronicos', 'Tecnologia')
         .when(F.col('categoria') == 'Vestuario',   'Moda')
         .when(F.col('categoria') == 'Moveis',      'Casa & Decoração')
         .when(F.col('categoria') == 'Casa',        'Casa & Decoração')
         .when(F.col('categoria') == 'Esportes',    'Bem-Estar')
         .when(F.col('categoria') == 'Beleza',      'Bem-Estar')
         .when(F.col('categoria') == 'Brinquedos',  'Entretenimento')
         .when(F.col('categoria') == 'Automotivo',  'Automotivo')
    )

    # 5. Marca temporal de ingestão
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação das colunas
    .select(
        'categoria',
        'segmento',
        'timestamp_ingestion',
    )

    .orderBy('categoria')
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

Transformações aplicadas. Schema resultante:
root
 |-- categoria: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)



In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────
# Confirmação do resultado para ter certeza que a transformação está correta.
# Espera-se exatamente 8 registros — um por categoria canônica — e que nenhum valor de segmento ficou nulo.

total            = df_silver.count()
segmentos_nulos  = df_silver.filter(F.col('segmento').isNull()).count()

print(f'Total de categorias canônicas : {total}')
print(f'Segmentos nulos (esperado = 0): {segmentos_nulos}')
print()
print('Conteúdo completo da dimensão:')
df_silver.show(truncate=False)

Total de categorias canônicas : 8
Segmentos nulos (esperado = 0): 0

Conteúdo completo da dimensão:
+-----------+----------------+--------------------------+
|categoria  |segmento        |timestamp_ingestion       |
+-----------+----------------+--------------------------+
|Automotivo |Automotivo      |2026-05-02 05:55:38.490186|
|Beleza     |Bem-Estar       |2026-05-02 05:55:38.490186|
|Brinquedos |Entretenimento  |2026-05-02 05:55:38.490186|
|Casa       |Casa & Decoração|2026-05-02 05:55:38.490186|
|Eletronicos|Tecnologia      |2026-05-02 05:55:38.490186|
|Esportes   |Bem-Estar       |2026-05-02 05:55:38.490186|
|Moveis     |Casa & Decoração|2026-05-02 05:55:38.490186|
|Vestuario  |Moda            |2026-05-02 05:55:38.490186|
+-----------+----------------+--------------------------+



In [0]:
# ─── Grava silver.dim_categorias_produto ─────────────────────────────────────
# Usa-se overwrite para garantir idempotência: A reexecução do notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.dim_categorias_produto')
)

print(f'{silver_schema}.dim_categorias_produto gravada com {df_silver.count()} registros.')

vcommerce_silver.dim_categorias_produto gravada com 8 registros.


In [0]:
# ─── Resumo final ─────────────────────────────────────────────────────────────

tabelas = [
    f'{silver_schema}.dim_categorias_produto',
]

print('=== Camada Silver — Categorias de Produto ===')
print(f'{"Tabela":<45} {"Linhas":>8} {"Colunas":>8}')
print('-' * 65)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<45} {df_t.count():>8,} {len(df_t.columns):>8}')

=== Camada Silver — Categorias de Produto ===
Tabela                                          Linhas  Colunas
-----------------------------------------------------------------
vcommerce_silver.dim_categorias_produto              8        3


## Tratamento: `dim_clientes`

**Origem:** `bronze.clientes`  
**Destino:** `silver.dim_clientes`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `nome`, `sobrenome` | Variações de case (`JOAO`, `joao`, `João`) | `INITCAP()` + `TRIM()` |
| `email` | Case misto (`Joao@Gmail.com`) | `LOWER()` + `TRIM()` |
| `genero` | ~10 variações para 3 categorias (`M`, `male`, `MASC`…) | Mapeamento canônico via cadeia de `WHEN` |
| `origem` | Variações (`app`, `APP`, `App Mobile`, `web`, `WEB`…) | `LOWER()` + `TRIM()` + mapeamento canônico |
| `cidade` | Case misto (`são paulo`, `SÃO PAULO`) | `INITCAP()` + `TRIM()` |
| `estado` | Case misto e abreviações inconsistentes | `UPPER()` + `TRIM()` |
| `pais` | Case misto | `UPPER()` + `TRIM()` |
| `data_nascimento`, `data_cadastro` | Inferidas como `string` na Bronze | Cast explícito para `date` |
| `telefone`, `endereco` | Possíveis espaços extras | `TRIM()` |
| `device_ids` | String bruta — possível array serializado | Mantido como `string` para análise futura |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | `ROW_NUMBER()` sobre `id_cliente` por `timestamp_ingestion DESC` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `nome_completo` | `CONCAT(nome, ' ', sobrenome)` |
| `idade` | `FLOOR(DATEDIFF(current_date, data_nascimento) / 365)` |
| `faixa_etaria` | `CASE` sobre `idade`: Menor de 18 / 18-24 / 25-34 / 35-44 / 45-59 / 60+ |
| `tempo_cliente_dias` | `DATEDIFF(current_date, data_cadastro)` |
| `regiao` | `CASE` sobre `estado` → Norte / Nordeste / Centro-Oeste / Sudeste / Sul |
| `timestamp_ingestion` | `current_timestamp()` — recriado no momento da carga Silver |

In [0]:
#Leitura da Bronze
#Usa o registro mais recente por id_cliente (maior timestamp_ingestion) para
#Neutralizar reprocessamentos da Bronze em mode=append.

df_raw = spark.table(f'{bronze_schema}.clientes')

window_dedup = Window.partitionBy('id_cliente').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   #remove timestamp_ingestion que será recriado ao final
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

Registros na Bronze  : 61,345
Após deduplicação    : 58,322
Duplicatas removidas : 3,023


In [0]:
#Mapeamentos canônicos
#Genero: 3 categorias reais fragmentadas em múltiplas variações
#Masculino  - m, masculino, male, masc, homem, h
#Feminino   - f, feminino, female, fem, mulher
#Outro      - outro, other, nb, nao_binario, nao informado
#Garantem que qualquer análise que for feita em cima de dados com esssas categorias sejam consistentes

genero_map = {
    'm'          : 'Masculino', 'masculino': 'Masculino', 'male' : 'Masculino',
    'masc'       : 'Masculino', 'homem'    : 'Masculino', 'h'    : 'Masculino',
    'f'          : 'Feminino',  'feminino' : 'Feminino',  'female': 'Feminino',
    'fem'        : 'Feminino',  'mulher'   : 'Feminino',
    'o'          : 'Outro',     'outro'    : 'Outro',     'other': 'Outro',
    'nb'         : 'Outro',     'nao_binario': 'Outro',
    'nao-informado' : 'Outro',
    'nao informado': 'Outro',
}

genero_expr = F.lit('Não informado')
for raw_val, canonical in genero_map.items():
    genero_expr = F.when(
        F.lower(F.trim(F.col('genero'))) == raw_val, canonical
    ).otherwise(genero_expr)

#origem: canônicas — App / Web / Parceiro / Não informado
origem_map = {
    'app'        : 'App',      'app mobile': 'App',    'mobile': 'App',
    'web'        : 'Web',      'website'   : 'Web',    'site'  : 'Web',
    'parceiro'   : 'Parceiro', 'partner'   : 'Parceiro',
    'indicação'  : 'Indicação', 'indicacao'  : 'Indicação', 'indicaçao': 'Indicação',
}

origem_expr = F.lit('Não informado')
for raw_val, canonical in origem_map.items():
    origem_expr = F.when(
        F.lower(F.trim(F.col('origem'))) == raw_val, canonical
    ).otherwise(origem_expr)

print('Expressões de mapeamento criadas.')

Expressões de mapeamento criadas.


In [0]:
#Principais transformações

regiao_expr = (
    #criação de coluna de agrupamento por região 
    

    F.when(F.upper(F.col('estado')).isin(
        'AMAZONAS','PARÁ','ACRE','RONDÔNIA','RORAIMA','AMAPÁ','TOCANTINS'
    ), 'Norte')
     .when(F.upper(F.col('estado')).isin(
        'BAHIA','SERGIPE','ALAGOAS','PERNAMBUCO','PARAÍBA',
        'RIO GRANDE DO NORTE','CEARÁ','PIAUÍ','MARANHÃO'
    ), 'Nordeste')
     .when(F.upper(F.col('estado')).isin(
        'MATO GROSSO','MATO GROSSO DO SUL','GOIÁS','DISTRITO FEDERAL'
    ), 'Centro-Oeste')
     .when(F.upper(F.col('estado')).isin(
        'SÃO PAULO','RIO DE JANEIRO','MINAS GERAIS','ESPÍRITO SANTO'
    ), 'Sudeste')
     .when(F.upper(F.col('estado')).isin(
        'PARANÁ','SANTA CATARINA','RIO GRANDE DO SUL'
    ), 'Sul')
     .otherwise('Não identificada')
)

df_silver = (
    df_dedup

    #Sanitização textual
    .withColumn('nome',      F.initcap(F.trim(F.col('nome'))))
    .withColumn('sobrenome', F.initcap(F.trim(F.col('sobrenome'))))
    .withColumn('email',     F.lower(F.trim(F.col('email'))))
    .withColumn('telefone',  F.trim(F.col('telefone')))
    .withColumn('endereco',  F.trim(F.col('endereco')))
    .withColumn('cidade',    F.initcap(F.trim(F.col('cidade'))))
    .withColumn('estado',    F.upper(F.trim(F.col('estado'))))
    .withColumn('pais',      F.upper(F.trim(F.col('pais'))))

    #Mapeamentos canônicos
    .withColumn('genero', genero_expr)
    .withColumn('origem', origem_expr)

    #Tipagem explícita
    .withColumn('data_nascimento', F.col('data_nascimento').cast('date'))
    .withColumn('data_cadastro',   F.col('data_cadastro').cast('date'))

    #Colunas derivadas
    .withColumn('nome_completo',
        F.concat_ws(' ', F.col('nome'), F.col('sobrenome'))
    )
    .withColumn('idade',
                F.when(
        F.datediff(F.current_date(), F.col('data_nascimento')) < 0, None
    ).otherwise(
        F.floor(F.datediff(F.current_date(), F.col('data_nascimento')) / 365).cast('int')
    ))
    .withColumn('faixa_etaria',
        F.when(F.col('idade') < 18,  'Menor de 18')
         .when(F.col('idade') <= 24, '18-24')
         .when(F.col('idade') <= 34, '25-34')
         .when(F.col('idade') <= 44, '35-44')
         .when(F.col('idade') <= 59, '45-59')
         .otherwise('60+')
    )
    .withColumn('tempo_cliente_dias',
        F.datediff(F.current_date(), F.col('data_cadastro'))
    )
    .withColumn('regiao', regiao_expr)

    #Rastreabilidade
    .withColumn('timestamp_ingestion', F.current_timestamp())

    #Ordenação de colunas para facilitar visualização
    .select(
        'id_cliente',
        'nome',
        'sobrenome',
        'nome_completo',
        'email',
        'telefone',
        'genero',
        'data_nascimento',
        'idade',
        'faixa_etaria',
        'data_cadastro',
        'tempo_cliente_dias',
        'endereco',
        'cidade',
        'estado',
        'regiao',
        'pais',
        'device_ids',
        'origem',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

Transformações aplicadas. Schema resultante:
root
 |-- id_cliente: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- sobrenome: string (nullable = true)
 |-- nome_completo: string (nullable = false)
 |-- email: string (nullable = true)
 |-- telefone: string (nullable = true)
 |-- genero: string (nullable = false)
 |-- data_nascimento: date (nullable = true)
 |-- idade: integer (nullable = true)
 |-- faixa_etaria: string (nullable = false)
 |-- data_cadastro: date (nullable = true)
 |-- tempo_cliente_dias: integer (nullable = true)
 |-- endereco: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- regiao: string (nullable = false)
 |-- pais: string (nullable = true)
 |-- device_ids: string (nullable = true)
 |-- origem: string (nullable = false)
 |-- timestamp_ingestion: timestamp (nullable = false)



In [0]:
#Validação pré-escrita 
#Exibir métricas do df antes de escrever no formato delta

total             = df_silver.count()
nulos_nascimento  = df_silver.filter(F.col('data_nascimento').isNull()).count()
nulos_cadastro    = df_silver.filter(F.col('data_cadastro').isNull()).count()
nao_informado_gen = df_silver.filter(F.col('genero') == 'Não informado').count()
nao_informado_ori = df_silver.filter(F.col('origem') == 'Não informado').count()

print(f'Total de clientes          : {total:,}')
print(f'Nulos em data_nascimento   : {nulos_nascimento:,}')
print(f'Nulos em data_cadastro     : {nulos_cadastro:,}')
print(f'Gênero "Não informado"     : {nao_informado_gen:,} ({nao_informado_gen/total*100:.1f}%)')
print(f'Origem "Não informado"     : {nao_informado_ori:,} ({nao_informado_ori/total*100:.1f}%)')
print()
print('Distribuição de gênero após normalização:')
df_silver.groupBy('genero').count().orderBy(F.desc('count')).show()
print('Distribuição de origem após normalização:')
df_silver.groupBy('origem').count().orderBy(F.desc('count')).show()
print('Distribuição de faixa_etaria:')
df_silver.groupBy('faixa_etaria').count().orderBy('faixa_etaria').show()

Total de clientes          : 58,322
Nulos em data_nascimento   : 0
Nulos em data_cadastro     : 0
Gênero "Não informado"     : 0 (0.0%)
Origem "Não informado"     : 0 (0.0%)

Distribuição de gênero após normalização:
+---------+-----+
|   genero|count|
+---------+-----+
| Feminino|31356|
|Masculino|26417|
|    Outro|  549|
+---------+-----+

Distribuição de origem após normalização:
+---------+-----+
|   origem|count|
+---------+-----+
|      Web|33984|
|      App|17465|
|Indicação| 6873|
+---------+-----+

Distribuição de faixa_etaria:
+------------+-----+
|faixa_etaria|count|
+------------+-----+
|       18-24| 6890|
|       25-34|16476|
|       35-44|16127|
|       45-59|13433|
|         60+| 4204|
| Menor de 18| 1192|
+------------+-----+



In [0]:
#Grava silver.dim_clientes
#overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.dim_clientes')
)

print(f'{silver_schema}.dim_clientes gravada com {df_silver.count():,} registros.')

vcommerce_silver.dim_clientes gravada com 58,322 registros.


---

## Tratamento: `dim_produtos` + `dim_categorias_produto`

**Origem:** `bronze.catalogo_produtos`  
**Destino:** `silver.dim_produtos`, `silver.dim_categorias_produto`

### Problemas identificados na Bronze

 Coluna | Problema | Tratamento |
---|---|---|
 `categoria` | 27 nulos + ~40 variações para 8 categorias reais (`cas@`, `Cas4`, `M0VEIS`, `3sportes`…) | Mapeamento canônico; nulos → `'Indefinida'` |
 `ativo` | 10 formas para verdadeiro/falso (`S`, `SIM`, `Sim`, `1`, `N`, `NAO`, `0`…) | Normalização para boolean |
 `preco` | 23 nulos + valores negativos (mín = -100.00) + valores zero | Nulos mantidos; valores ≤ 0 → `null` |
 `avaliacao_interna` | 100% nulo | Coluna removida |
 `peso_kg` / `estoque_disponivel` | 23 nulos cada | Mantidos como `null` |

### Colunas derivadas

 Coluna | Lógica |
---|---|
 `categoria` | Normalizado de ~40 variações para 8 valores canônicos: `Eletronicos`, `Moveis`, `Esportes`, `Beleza`, `Automotivo`, `Brinquedos`, `Casa`, `Vestuario` |
 `ativo` | Normalizado de 10 variações textuais para `boolean` — `S/SIM/1 → true`, `N/NAO/0 → false` |
 `preco` | Valores negativos ou zero corrigidos: `preco ≤ 0` vira `null` |

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────

df_prod_raw = spark.table(f'{bronze_schema}.catalogo_produtos')

window_prod = Window.partitionBy('id_produto').orderBy(F.desc('timestamp_ingestion'))

df_prod = (
    df_prod_raw
    .withColumn('_rank', F.row_number().over(window_prod))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')
)

print(f'Bronze: {df_prod_raw.count():,} registros -> após dedup: {df_prod.count():,}')

Bronze: 517 registros -> após dedup: 517


In [0]:
# ─── Mapeamento canônico de categoria ────────────────────────────────────────
# 8 categorias reais fragmentadas em ~40 variações na Bronze.

categoria_map = {
    # Eletronicos
    'eletronicos':'Eletronicos', 'eletronico':'Eletronicos',
    'eletrônico':'Eletronicos',  'elet':'Eletronicos',
    # Moveis
    'moveis':'Moveis', 'móveis':'Moveis', 'm0veis':'Moveis', 'mov':'Moveis',
    # Esportes
    'esportes':'Esportes', 'esporte':'Esportes', '3sportes':'Esportes',
    'esport':'Esportes',   'esp':'Esportes',
    # Beleza
    'beleza':'Beleza', 'b3leza':'Beleza', 'belz':'Beleza', 'bel':'Beleza',
    # Automotivo
    'automotivo':'Automotivo', 'automotiv3':'Automotivo',
    'autom':'Automotivo',      'aut':'Automotivo',
    # Brinquedos
    'brinquedos':'Brinquedos', 'brinquedo':'Brinquedos',
    'br1nquedos':'Brinquedos', 'brinq':'Brinquedos', 'brin':'Brinquedos',
    # Casa
    'casa':'Casa', 'cas@':'Casa', 'cas4':'Casa', 'cas':'Casa',
    # Vestuario
    'vestuario':'Vestuario', 'vestuarios':'Vestuario',
    'vest':'Vestuario',      'vestu':'Vestuario',
}

cat_expr = F.col('categoria')
for raw_val, canonical in categoria_map.items():
    cat_expr = F.when(
        F.lower(F.col('categoria')) == raw_val, canonical
    ).otherwise(cat_expr)

cat_expr = F.when(
    F.lower(F.col('categoria')).isin(list(categoria_map.keys())), cat_expr
).otherwise(F.lit('Indefinida'))

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────
# Problemas tratados:
#   categoria        → mapeamento canônico (~40 variações → 8 valores)
#   ativo            → normalização para boolean (10 variações textuais)
#   preco            → limpeza de formatação (remove 'R$', espaços), conversão , → ., try_cast para tratar inválidos, ABS para negativos
#   avaliacao_interna→ removida (100% nulo, sem valor analítico)

ativo_true  = ['s', 'sim', '1']
ativo_false = ['n', 'nao', '0']

df_dim_produtos = (
    df_prod

    # 1. Categoria normalizada
    .withColumn('categoria', cat_expr)

    # 2. Ativo → boolean
    .withColumn('ativo',
        F.when(F.lower(F.col('ativo')).isin(ativo_true),  F.lit(True))
         .when(F.lower(F.col('ativo')).isin(ativo_false), F.lit(False))
         .otherwise(F.lit(None).cast('boolean')))

    # 3. Preço: limpa formatação de moeda, try_cast para tolerar inválidos, e ABS para negativos
    .withColumn('preco',
        F.when(
            F.expr(
                "try_cast(regexp_replace(regexp_replace(preco, 'R\\$\\s*', ''), ',', '.') as decimal(10,2))"
            ).isNotNull() &
            (F.expr(
                "try_cast(regexp_replace(regexp_replace(preco, 'R\\$\\s*', ''), ',', '.') as decimal(10,2))"
            ) <= 0),
            F.lit(None).cast('decimal(10,2)')
        ).otherwise(
            F.abs(
                F.expr(
                    "try_cast(regexp_replace(regexp_replace(preco, 'R\\$\\s*', ''), ',', '.') as decimal(10,2))"
                )
            )
        )
    )

    # 4. Tipos numéricos com try_cast para tolerar valores inválidos ('null', strings vazias, etc.)
    .withColumn('peso_kg', F.expr("try_cast(peso_kg as decimal(8,2))"))
    .withColumn('estoque_disponivel', F.expr("try_cast(estoque_disponivel as int)"))
    .withColumn('data_cadastro_produto', F.to_date('data_cadastro_produto'))

    # 5. Remove coluna 100% nula
    .drop('avaliacao_interna')

    # 6. Timestamp desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    .select(
        'id_produto', 'nome_produto', 'categoria', 'preco',
        'fornecedor', 'peso_kg', 'estoque_disponivel',
        'ativo', 'data_cadastro_produto', 'timestamp_ingestion',
    )
)

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total_prod = df_dim_produtos.count()
indef      = df_dim_produtos.filter(F.col('categoria') == 'Indefinida').count()
preco_nulo = df_dim_produtos.filter(F.col('preco').isNull()).count()
ativos     = df_dim_produtos.filter(F.col('ativo') == True).count()
inativos   = df_dim_produtos.filter(F.col('ativo') == False).count()

print(f'Total de produtos      : {total_prod:,}')
print(f'Categoria "Indefinida" : {indef:,}  ({indef/total_prod*100:.1f}%)')
print(f'Preco nulo             : {preco_nulo:,}')
print(f'Ativos                 : {ativos:,}  ({ativos/total_prod*100:.1f}%)')
print(f'Inativos               : {inativos:,}  ({inativos/total_prod*100:.1f}%)')
print()
df_dim_produtos.groupBy('categoria').count().orderBy(F.desc('count')).show()

Total de produtos      : 517
Categoria "Indefinida" : 28  (5.4%)
Preco nulo             : 89
Ativos                 : 326  (63.1%)
Inativos               : 107  (20.7%)

+-----------+-----+
|  categoria|count|
+-----------+-----+
| Brinquedos|   64|
|Eletronicos|   63|
|     Moveis|   62|
|  Vestuario|   61|
|       Casa|   61|
|     Beleza|   60|
|   Esportes|   60|
| Automotivo|   58|
| Indefinida|   28|
+-----------+-----+



In [0]:
# ─── Grava silver.dim_produtos ───────────────────────────────────────────────

(
    df_dim_produtos.write
                   .format('delta')
                   .mode('overwrite')
                   .option('overwriteSchema', 'true')
                   .saveAsTable(f'{silver_schema}.dim_produtos')
)

print(f'{silver_schema}.dim_produtos gravada com {df_dim_produtos.count():,} registros.')

vcommerce_silver.dim_produtos gravada com 517 registros.


In [0]:
# ─── dim_categorias_produto ───────────────────────────────────────────────────

df_dim_cat = (
    spark.table(f'{silver_schema}.dim_produtos')
         .select('categoria').distinct()
         .withColumn('categoria_normalizada',
             F.lower(
                 F.regexp_replace(
                     F.translate('categoria', 'áàãâéêíóôõúüç', 'aaaaeeiooouuc'),
                     '[^a-z0-9]', '_'
                 )
             ))
         .orderBy('categoria')
)

df_dim_cat.show(truncate=False)

(
    df_dim_cat.write
              .format('delta')
              .mode('overwrite')
              .option('overwriteSchema', 'true')
              .saveAsTable(f'{silver_schema}.dim_categorias_produto')
)

print(f'{silver_schema}.dim_categorias_produto gravada.')

+-----------+---------------------+
|categoria  |categoria_normalizada|
+-----------+---------------------+
|Automotivo |_utomotivo           |
|Beleza     |_eleza               |
|Brinquedos |_rinquedos           |
|Casa       |_asa                 |
|Eletronicos|_letronicos          |
|Esportes   |_sportes             |
|Indefinida |_ndefinida           |
|Moveis     |_oveis               |
|Vestuario  |_estuario            |
+-----------+---------------------+

vcommerce_silver.dim_categorias_produto gravada.


In [0]:
df_preco_zero_null = spark.table(f'{silver_schema}.dim_produtos').filter(
    (F.col('preco') <= 0) | (F.col('preco').isNull())
)
display(df_preco_zero_null)

id_produto,nome_produto,categoria,preco,fornecedor,peso_kg,estoque_disponivel,ativo,data_cadastro_produto,timestamp_ingestion
PROD-0005,Notebook Intel Core i7,Eletronicos,null,TechBrasil Distribuidora,3.77,271,true,2018-12-02,2026-05-02T05:56:01.956Z
PROD-0009,Smart TV 55 Polegadas,Eletronicos,null,Global Electronics Ltda,2.80,459,null,2024-02-04,2026-05-02T05:56:01.956Z
PROD-0010,Smart TV 32 Polegadas,Eletronicos,null,Global Electronics Ltda,3.38,71,true,2020-10-23,2026-05-02T05:56:01.956Z
PROD-0013,Fone de Ouvido Bluetooth,Eletronicos,null,TechBrasil Distribuidora,0.84,90,null,2025-11-08,2026-05-02T05:56:01.956Z
PROD-0016,Caixa de Som Bluetooth,Eletronicos,null,Shenzhen Connect Brasil,4.20,37,true,2022-10-14,2026-05-02T05:56:01.956Z
PROD-0032,Headset Bluetooth,Eletronicos,null,Alpha Tech Importados,2.21,71,true,2026-02-20,2026-05-02T05:56:01.956Z
PROD-0045,Hub USB 7 Portas,Eletronicos,null,Shenzhen Connect Brasil,3.49,177,true,2019-02-21,2026-05-02T05:56:01.956Z
PROD-0048,Bateria Portátil 20000mAh,Eletronicos,null,Shenzhen Connect Brasil,0.75,493,true,2024-03-18,2026-05-02T05:56:01.956Z
PROD-0052,Impressora Térmica,Eletronicos,null,TechBrasil Distribuidora,2.76,384,true,2023-06-12,2026-05-02T05:56:01.956Z
PROD-0063,Caixinha de Som Mini,Eletronicos,null,Global Electronics Ltda,4.06,4,true,2024-10-16,2026-05-02T05:56:01.956Z


In [0]:
df_produto = spark.table(f'{silver_schema}.dim_produtos').filter(F.col('id_produto') == 'PROD-0514')
display(df_produto)

id_produto,nome_produto,categoria,preco,fornecedor,peso_kg,estoque_disponivel,ativo,data_cadastro_produto,timestamp_ingestion
PROD-0514,Conjunto Esportivo Feminino,Vestuario,null,Fashion Import Ltda,1.09,233,false,2025-11-25,2026-05-02T05:56:01.956Z


In [0]:
total_produtos_silver = spark.table(f'{silver_schema}.dim_produtos').count()
print(f'Quantidade de linhas em produtos silver: {total_produtos_silver}')

Quantidade de linhas em produtos silver: 517


---

## Tratamento: `ft_pedidos`

**Origem:** `bronze.pedidos`  
**Destino:** `silver.ft_pedidos`

### Análise inicial da Bronze

Na exploração inicial da tabela `bronze.pedidos`, não foram encontrados registros duplicados por `id_pedido`, duplicatas exatas, valores nulos ou campos vazios aparentes. Ainda assim, será mantida uma deduplicação preventiva por `id_pedido`, priorizando o registro mais recente por `timestamp_ingestion`.

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `valor_pedido` | Valores em string com presença de símbolos monetários (`R`, `r`, `$`), espaços, vírgula decimal e valores negativos | Remover caracteres indesejados, converter para `decimal(10,2)`, sinalizar linhas com valores originalmente negativos com `flag_valor_pedido_negativo` e anular valores negativos |
| `data_pedido` | Datas em string com separadores e formatos diferentes (`yyyy-MM-dd`, `yyyy-dd-MM`, `MM-dd-yyyy`, `dd-MM-yyyy`, `dd-yyyy-MM`, `MM-yyyy-dd`) e casos ambíguos como `2025-03-09`, que podem representar `yyyy-MM-dd` ou `yyyy-dd-MM` | Padronizar separadores, reorganizar ano/mês/dia, aplicar o padrão `yyyy-MM-dd`, converter para `date` e sinalizar datas possivelmente ambíguas |
| `metodo_pagamento` | 15 variações (`PIX`, `pIx`, `P1X`, `pix`, `Pix`, `Cartao`, `CARTAO`, `c@rtao`, `cartao`, `Cartão`, `crt`, `Boleto`, `BOLETO`, `bol`, `b0leto`) | Normalizar para valores canônicos (`Pix`, `Cartão`, `Boleto`) |
| `status` | 23 variações (`APR`, `APROVADO`, `Aprovado`, `Aprovadoo`, `aprov`, `aprovado`, `REC`, `RECUSADO`, `Recusado`, `Recusadoo`, `recus`, `recusado`, `REEMB`, `REEMBOLSADO`, `Reembolsad`, `Reembolsado`, `reembolso`, `reembolsado`, `PROC`, `PROCESSANDO`, `Processando`, `process`, `processando`) | Normalizar para valores canônicos (`Aprovado`, `Recusado`, `Reembolsado`, `Processando`) |
| `quantidade` | Valores em string com textos numéricos (`um`, `quatro`, `cinco`), números decimais (`2.0`, `3.0`, `4.0`) e quantidades menores ou iguais a zero | Converter para inteiro, mapear textos conhecidos, transformar decimais em inteiros, anular valores menores ou iguais a zero e sinalizar com `flag_quantidade_menor_igual_a_zero` |
| Duplicatas | Não foram encontradas duplicatas na exploração inicial, mas reprocessamentos futuros da Bronze podem gerar múltiplas versões do mesmo pedido por conta do `mode=append` | Deduplicação preventiva por `id_pedido`, mantendo o registro com maior `timestamp_ingestion` |

### Colunas de flags criadas

| Coluna | Lógica |
|---|---|
| `flag_valor_pedido_negativo` | Indica registros em que `valor_pedido` era negativo antes do tratamento; esses casos foram anulados para evitar distorções financeiras |
| `flag_data_pedido_ambigua` | Indica registros em que os elementos de mês e dia são ambos menores ou iguais a 12, podendo gerar dupla interpretação da data original |
| `flag_quantidade_menor_igual_a_zero` | Indica registros em que `quantidade` era menor ou igual a zero antes da anulação |
| `flag_pedido_com_inconsistencia` | Indica registros que apresentaram ao menos uma inconsistência identificada, considerando data ambígua, valor negativo na origem, quantidade inválida na origem, campos nulos após tratamento ou categorias classificadas como `Outro` |

### Coluna de enriquecimento criada

| Coluna | Lógica |
|---|---|
| `ano_mes` | Ano e mês extraídos de `data_pedido`, no formato `yyyy-MM`, para facilitar análises e agregações mensais |

### Observações

- O `timestamp_ingestion` original da Bronze é usado apenas para deduplicação. Após essa etapa, ele é removido e recriado na Silver com o instante da carga tratada.
- Foi avaliada a possibilidade de calcular `valor_total` a partir do preço do catálogo de produtos multiplicado pela quantidade. Porém, foi identificada divergência massiva entre esse cálculo e o valor informado em `valor_pedido`. Por isso, nesta etapa, `valor_pedido` foi preservado como o valor registrado na origem, após limpeza e validação. A criação de `valor_total` deve ser definida posteriormente com regra de negócio clara ou tratada na camada Gold.
- Datas consideradas ambíguas não estão necessariamente erradas, até porque estão sendo tratadas no formato certo, mas não é correto relevar o fato de que elas podem ter sido interpretadas de forma diferente da intenção original. Por isso, esses casos foram mantidos na base tratada e sinalizados por meio de uma flag de qualidade, permitindo rastreabilidade e análise posterior sem descartar registros potencialmente válidos.
- A coluna `flag_pedido_com_inconsistencia` consolida os principais sinais de qualidade do registro, permitindo identificar rapidamente pedidos que exigem cautela em análises posteriores, sem remover registros potencialmente úteis da base.

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Lê todos os registros brutos da tabela de pedidos ingerida na camada Bronze.
# Como a Bronze usa mode=append, reprocessamentos podem gerar registros duplicados
# para o mesmo id_pedido. Usamos row_number() sobre a janela particionada por
# id_pedido (ordenada pelo timestamp_ingestion mais recente) para ficar apenas
# com a versão mais atual de cada pedido.
# O timestamp_ingestion da Bronze é removido após a deduplicação e será recriado
# com o instante da carga Silver na transformação principal.

df_raw = spark.table(f'{bronze_schema}.pedidos')

window_dedup = Window.partitionBy('id_pedido').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')
)

total_raw = df_raw.count()
total_dedup = df_dedup.count()

print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

Registros na Bronze  : 314,900
Após deduplicação    : 314,900
Duplicatas removidas : 0


In [0]:
# ─── Regra de tratamento de valor_pedido ──────────────────────────────────────
# A coluna valor_pedido veio como string, com presença de símbolos monetários,
# espaços, vírgula decimal e valores negativos.
#
# Fluxo aplicado:
# 1. Remover espaços desnecessários
# 2. Remover símbolo monetário R$
# 3. Remover caracteres indesejados
# 4. Padronizar vírgula decimal para ponto
# 5. Converter o valor para decimal(10,2)
# 6. Criar flag para indicar valores negativos
# 7. Anular valores negativos para evitar distorção em análises financeiras

valor_pedido_limpo_expr = (
    F.regexp_replace(
        F.regexp_replace(
            F.regexp_replace(
                F.lower(F.trim(F.col('valor_pedido').cast('string'))),
                'r\\$\\s*',
                ''
            ),
            '[^0-9,.-]',
            ''
        ),
        ',',
        '.'
    )
)

valor_pedido_num_expr = F.expr("""
    try_cast(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    lower(trim(cast(valor_pedido as string))),
                    'r\\\\$\\\\s*',
                    ''
                ),
                '[^0-9,.-]',
                ''
            ),
            ',',
            '.'
        ) as decimal(10,2)
    )
""")

flag_valor_pedido_negativo_expr = (
    valor_pedido_num_expr < 0
)

valor_pedido_expr = (
    F.when(flag_valor_pedido_negativo_expr, F.lit(None).cast('decimal(10,2)'))
     .otherwise(valor_pedido_num_expr)
)

print('Expressões de tratamento de valor_pedido criadas.')

Expressões de tratamento de valor_pedido criadas.


In [0]:
# ─── Regra de tratamento de data_pedido ───────────────────────────────────────
# A coluna data_pedido veio como string, com separadores e formatos diferentes.
# Foram identificados registros com separadores como "/", "-" e ".", além de
# variações na posição do ano, mês e dia, como yyyy-MM-dd, yyyy-dd-MM,
# MM-dd-yyyy, dd-MM-yyyy, dd-yyyy-MM e MM-yyyy-dd.
#
# Por causa dessa variação, algumas datas podem ser interpretadas de mais de uma
# forma. Por exemplo, 2025-03-09 pode representar 9 de março de 2025
# (yyyy-MM-dd) ou 3 de setembro de 2025 (yyyy-dd-MM). Nesses casos, a ordem
# original é preservada e uma flag é criada para indicar a ambiguidade.
#
# Fluxo aplicado:
# 1. Padronizar os separadores para "-"
# 2. Separar a data em três partes
# 3. Identificar em qual posição está o ano (yyyy)
# 4. Definir mês e dia:
#    - se um dos elementos for maior que 12, ele é considerado dia
#    - se ambos forem menores ou iguais a 12, a ordem original é mantida
# 5. Montar a data no padrão yyyy-MM-dd
# 6. Converter a data padronizada para o tipo date
# 7. Criar uma flag para indicar registros com data ambígua

# 1. Padronização dos separadores
# Troca "/" e "." por "-", garantindo uma estrutura única para quebrar a data.
data_norm_expr = F.regexp_replace(
    F.regexp_replace(
        F.trim(F.col('data_pedido').cast('string')),
        '/',
        '-'
    ),
    '\\.',
    '-'
)

# 2. Separação da data em três partes
# Exemplo: 2025-02-26 vira p1=2025, p2=02, p3=26.
partes_data = F.split(data_norm_expr, '-')

p1 = partes_data.getItem(0)
p2 = partes_data.getItem(1)
p3 = partes_data.getItem(2)

p1_int = p1.cast('int')
p2_int = p2.cast('int')
p3_int = p3.cast('int')

# 3. Identificação da posição do ano
# O ano é identificado pela parte com 4 dígitos.
ano_inicio = F.length(p1) == 4
ano_meio = F.length(p2) == 4
ano_fim = F.length(p3) == 4

ano_expr = (
    F.when(ano_inicio, p1)
     .when(ano_meio, p2)
     .when(ano_fim, p3)
)

# 4. Definição do mês
# A regra usa a posição do ano e os valores das outras duas partes.
# Se uma parte for maior que 12, ela não pode ser mês, então é tratada como dia.
# Quando não há valor maior que 12, a ordem original é preservada.
mes_expr = (
    # Ano no início: yyyy-MM-dd ou yyyy-dd-MM
    F.when(ano_inicio & (p2_int > 12), p3)
     .when(ano_inicio, p2)

    # Ano no meio: dd-yyyy-MM ou MM-yyyy-dd
     .when(ano_meio & (p1_int > 12), p3)
     .when(ano_meio & (p3_int > 12), p1)
     .when(ano_meio, p1)

    # Ano no fim: dd-MM-yyyy ou MM-dd-yyyy
     .when(ano_fim & (p1_int > 12), p2)
     .when(ano_fim & (p2_int > 12), p1)
     .when(ano_fim, p1)
)

# 5. Definição do dia
# Usa a lógica complementar à definição do mês.
dia_expr = (
    # Ano no início: yyyy-MM-dd ou yyyy-dd-MM
    F.when(ano_inicio & (p2_int > 12), p2)
     .when(ano_inicio, p3)

    # Ano no meio: dd-yyyy-MM ou MM-yyyy-dd
     .when(ano_meio & (p1_int > 12), p1)
     .when(ano_meio & (p3_int > 12), p3)
     .when(ano_meio, p3)

    # Ano no fim: dd-MM-yyyy ou MM-dd-yyyy
     .when(ano_fim & (p1_int > 12), p1)
     .when(ano_fim & (p2_int > 12), p2)
     .when(ano_fim, p2)
)

# 6. Montagem da data padronizada
# Reorganiza ano, mês e dia no padrão yyyy-MM-dd.
# lpad garante mês/dia com dois dígitos, exemplo: 2025-2-6 vira 2025-02-06.
data_pedido_padronizada_expr = (
    F.when(
        ano_expr.isNotNull() & mes_expr.isNotNull() & dia_expr.isNotNull(),
        F.concat(
            ano_expr.cast('string'),
            F.lit('-'),
            F.lpad(mes_expr.cast('string'), 2, '0'),
            F.lit('-'),
            F.lpad(dia_expr.cast('string'), 2, '0')
        )
    )
)

# 7. Conversão para date
# A coluna final data_pedido será do tipo date e será exibida no padrão yyyy-MM-dd.
data_pedido_expr = F.to_date(data_pedido_padronizada_expr, 'yyyy-MM-dd')

# 8. Flag de ambiguidade
# Sinaliza registros em que os elementos de mês e dia são ambos <= 12.
# Nesses casos, os dois valores poderiam representar mês ou dia na data original.
flag_data_pedido_ambigua_expr = (
    mes_expr.cast('int').between(1, 12) &
    dia_expr.cast('int').between(1, 12)
)

print('Expressões de tratamento de data_pedido criadas.')

Expressões de tratamento de data_pedido criadas.


In [0]:
# ─── Mapeamento canônico de metodo_pagamento ──────────────────────────────────
# Na Bronze foram encontradas variações de escrita, caixa, abreviações e typos
# para 3 métodos de pagamento reais:
#
#   Pix     - PIX, pIx, P1X, pix, Pix
#   Cartão  - Cartao, CARTAO, c@rtao, cartao, Cartão, crt
#   Boleto  - Boleto, BOLETO, bol, b0leto
#
# Estratégia: normalizar para lowercase/trim e aplicar mapeamento direto.
# Valores não mapeados serão marcados como 'Outro' na transformação principal.

metodo_pagamento_map = {
    # Pix
    'pix'    : 'Pix',
    'p1x'    : 'Pix',
    # Cartão
    'cartao' : 'Cartão',
    'cartão' : 'Cartão',
    'c@rtao' : 'Cartão',
    'crt'    : 'Cartão',
    # Boleto
    'boleto' : 'Boleto',
    'bol'    : 'Boleto',
    'b0leto' : 'Boleto',
}

metodo_pagamento_mapping_expr = F.create_map([
    item
    for key_value in metodo_pagamento_map.items()
    for item in (F.lit(key_value[0]), F.lit(key_value[1]))
])

metodo_pagamento_expr = F.coalesce(
    metodo_pagamento_mapping_expr[F.lower(F.trim(F.col('metodo_pagamento').cast('string')))],
    F.lit('Outro')
)

print('Expressão de mapeamento de metodo_pagamento criada.')
print(f'Métodos mapeados: {sorted(set(metodo_pagamento_map.values()))}')

Expressão de mapeamento de metodo_pagamento criada.
Métodos mapeados: ['Boleto', 'Cartão', 'Pix']


In [0]:
# ─── Mapeamento canônico de status ────────────────────────────────────────────
# Na Bronze foram encontradas variações de escrita, caixa, abreviações e typos
# para 4 status reais:
#
#   Aprovado     - Aprovado, Aprovadoo, APROVADO, aprovado, aprov, APR
#   Recusado     - Recusado, recusado, RECUSADO, recus, REC, Recusadoo
#   Reembolsado  - Reembolsado, reembolsado, REEMBOLSADO, reembolso, REEMB,
#                  Reembolsad
#   Processando  - Processando, processando, PROCESSANDO, process, PROC
#
# Estratégia: normalizar para lowercase/trim e aplicar mapeamento direto.
# Valores não mapeados serão marcados como 'Outro' na transformação principal.

status_map = {
    # Aprovado
    'aprovado'   : 'Aprovado',
    'aprovadoo'  : 'Aprovado',
    'aprov'      : 'Aprovado',
    'apr'        : 'Aprovado',
    # Recusado
    'recusado'   : 'Recusado',
    'recusadoo'  : 'Recusado',
    'recus'      : 'Recusado',
    'rec'        : 'Recusado',
    # Reembolsado
    'reembolsado': 'Reembolsado',
    'reembolsad' : 'Reembolsado',
    'reembolso'  : 'Reembolsado',
    'reemb'      : 'Reembolsado',
    # Processando
    'processando': 'Processando',
    'process'    : 'Processando',
    'proc'       : 'Processando',
}

status_mapping_expr = F.create_map([
    item
    for key_value in status_map.items()
    for item in (F.lit(key_value[0]), F.lit(key_value[1]))
])

status_expr = F.coalesce(
    status_mapping_expr[F.lower(F.trim(F.col('status').cast('string')))],
    F.lit('Outro')
)

print('Expressão de mapeamento de status criada.')
print(f'Status mapeados: {sorted(set(status_map.values()))}')

Expressão de mapeamento de status criada.
Status mapeados: ['Aprovado', 'Processando', 'Recusado', 'Reembolsado']


In [0]:
# ─── Regra de tratamento de quantidade ────────────────────────────────────────
# A coluna quantidade veio como string, com valores numéricos, textos numéricos,
# números decimais e quantidades inválidas.
#
# Fluxo aplicado:
# 1. Remover espaços e padronizar o texto para lowercase
# 2. Mapear textos numéricos conhecidos para seus respectivos números
# 3. Converter valores decimais como 2.0, 3.0 e 4.0 para inteiro
# 4. Criar flag para indicar quantidades menores ou iguais a zero
# 5. Anular quantidades menores ou iguais a zero para evitar distorções analíticas

quantidade_str = F.lower(F.trim(F.col('quantidade').cast('string')))

quantidade_texto_map = {
    'um': '1',
    'quatro': '4',
    'cinco': '5'
}

quantidade_texto_mapping_expr = F.create_map([
    item
    for key_value in quantidade_texto_map.items()
    for item in (F.lit(key_value[0]), F.lit(key_value[1]))
])

quantidade_normalizada_expr = F.coalesce(
    quantidade_texto_mapping_expr[quantidade_str],
    quantidade_str
)

quantidade_num_expr = (
    quantidade_normalizada_expr
    .cast('double')
    .cast('int')
)

flag_quantidade_menor_igual_a_zero_expr = (
    quantidade_num_expr <= 0
)

quantidade_expr = (
    F.when(flag_quantidade_menor_igual_a_zero_expr, F.lit(None).cast('int'))
     .otherwise(quantidade_num_expr)
)

print('Expressões de tratamento de quantidade criadas.')

Expressões de tratamento de quantidade criadas.


In [0]:
# ─── Transformação principal: ft_pedidos ──────────────────────────────────────
# Aplica as regras de padronização e qualidade definidas anteriormente:
# - status normalizado
# - metodo_pagamento normalizado
# - data_pedido convertida para date
# - valor_pedido convertido para decimal
# - quantidade convertida para inteiro
# - flags de qualidade para datas ambíguas, valores negativos e quantidades inválidas
# - flag consolidada indicando pedidos com alguma inconsistência identificada
# - ano_mes criado a partir de data_pedido
# - timestamp_ingestion recriado com o instante da carga Silver

df_silver = (
    df_dedup

    # Flags de qualidade individuais
    .withColumn('flag_data_pedido_ambigua', flag_data_pedido_ambigua_expr)
    .withColumn('flag_valor_pedido_negativo', flag_valor_pedido_negativo_expr)
    .withColumn('flag_quantidade_menor_igual_a_zero', flag_quantidade_menor_igual_a_zero_expr)

    # Padronizações e conversões
    .withColumn('status', status_expr)
    .withColumn('metodo_pagamento', metodo_pagamento_expr)
    .withColumn('data_pedido', data_pedido_expr)
    .withColumn('valor_pedido', valor_pedido_expr)
    .withColumn('quantidade', quantidade_expr)

    # Flag consolidada de qualidade do registro
    .withColumn(
        'flag_pedido_com_inconsistencia',
        (
            F.col('flag_data_pedido_ambigua') |
            F.col('flag_valor_pedido_negativo') |
            F.col('flag_quantidade_menor_igual_a_zero') |
            F.col('data_pedido').isNull() |
            F.col('valor_pedido').isNull() |
            F.col('quantidade').isNull() |
            (F.col('status') == 'Outro') |
            (F.col('metodo_pagamento') == 'Outro')
        )
    )

    # Coluna de enriquecimento
    .withColumn('ano_mes', F.date_format(F.col('data_pedido'), 'yyyy-MM'))

    # Timestamp da carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # Organização final das colunas
    .select(
        'id_pedido',
        'id_cliente',
        'id_produto',
        'valor_pedido',
        'data_pedido',
        'ano_mes',
        'metodo_pagamento',
        'status',
        'quantidade',
        'flag_data_pedido_ambigua',
        'flag_valor_pedido_negativo',
        'flag_quantidade_menor_igual_a_zero',
        'flag_pedido_com_inconsistencia',
        'timestamp_ingestion'
    )
)

print('Tabela ft_pedidos montada.')

Tabela ft_pedidos montada.


In [0]:
# ─── Validação pré-escrita: ft_pedidos ────────────────────────────────────────
# Verifica os principais pontos de qualidade antes da gravação na camada Silver.

total_silver = df_silver.count()

qtd_pedidos_com_inconsistencia = df_silver.filter(
    F.col('flag_pedido_com_inconsistencia') == True
).count()

perc_pedidos_com_inconsistencia = (
    qtd_pedidos_com_inconsistencia / total_silver * 100
    if total_silver > 0
    else 0
)

print('Validação pré-escrita - ft_pedidos')
print('--------------------------------------------------')
print(f'Registros finais: {total_silver:,}')
print(f"Status como Outro: {df_silver.filter(F.col('status') == 'Outro').count():,}")
print(f"Métodos como Outro: {df_silver.filter(F.col('metodo_pagamento') == 'Outro').count():,}")
print(f"Datas nulas: {df_silver.filter(F.col('data_pedido').isNull()).count():,}")
print(f"Valores negativos remanescentes: {df_silver.filter(F.col('valor_pedido') < 0).count():,}")
print(f"Quantidades inválidas remanescentes: {df_silver.filter(F.col('quantidade') <= 0).count():,}")
print(f"Pedidos com inconsistência: {qtd_pedidos_com_inconsistencia:,} ({perc_pedidos_com_inconsistencia:.2f}%)")

display(df_silver.limit(10))

Validação pré-escrita - ft_pedidos
--------------------------------------------------
Registros finais: 314,900
Status como Outro: 0
Métodos como Outro: 0
Datas nulas: 0
Valores negativos remanescentes: 0
Quantidades inválidas remanescentes: 0
Pedidos com inconsistência: 141,511 (44.94%)


id_pedido,id_cliente,id_produto,valor_pedido,data_pedido,ano_mes,metodo_pagamento,status,quantidade,flag_data_pedido_ambigua,flag_valor_pedido_negativo,flag_quantidade_menor_igual_a_zero,flag_pedido_com_inconsistencia,timestamp_ingestion
0000336c-e8de-435d-876e-df51705f648f,7b652fe2-c341-47a7-9830-0d4f3c726c03,PROD-0430,550.66,2024-11-11,2024-11,Pix,Aprovado,2,true,false,false,true,2026-05-02T05:56:31.184Z
00004bc0-48c9-449b-afe1-ac4cd12a7fb7,db8ec0fe-417a-4ce7-a01c-a4c4a26996fa,PROD-0022,7120.15,2024-08-17,2024-08,Boleto,Aprovado,5,false,false,false,false,2026-05-02T05:56:31.184Z
000052ea-330e-4dd5-8b18-aad071cc0ff7,3ade8bd1-f696-4fc6-8719-b679d60728f1,PROD-0183,78.37,2023-08-02,2023-08,Cartão,Aprovado,5,true,false,false,true,2026-05-02T05:56:31.184Z
0000a4e5-4d16-491e-9048-c8206e0de355,3485c080-d848-4b19-900d-dfa497649bc0,PROD-0475,2763.00,2024-09-13,2024-09,Boleto,Aprovado,5,false,false,false,false,2026-05-02T05:56:31.184Z
0000dfe9-feb3-4d71-a8fb-0e6027b98757,eec41a54-b1bf-4ad3-9714-d16c17393b20,PROD-0363,44.85,2025-12-24,2025-12,Pix,Aprovado,null,false,false,true,true,2026-05-02T05:56:31.184Z
0000e299-1c75-43cf-aecd-e0cc4ca1ab62,e34b24fe-4f6e-419a-8316-60715d0bcdde,PROD-0294,264.86,2025-02-06,2025-02,Pix,Aprovado,4,true,false,false,true,2026-05-02T05:56:31.184Z
0000ed54-0e64-4eb9-8cb9-06b2db6027e4,a07f463f-18d0-4f8c-b03f-34be900f0ce5,PROD-0208,454.75,2023-08-01,2023-08,Pix,Aprovado,3,true,false,false,true,2026-05-02T05:56:31.184Z
0001289f-f61e-59d3-a33b-53b4690e0747,455aad53-0c41-4b7a-874c-fdab690f6af6,PROD-0214,109.62,2026-03-13,2026-03,Pix,Aprovado,2,false,false,false,false,2026-05-02T05:56:31.184Z
000210fe-fa9e-4c4d-95aa-27435d7dd7a6,12dd63d3-bda5-422c-8463-fcf052383b46,PROD-0167,242.99,2024-03-19,2024-03,Boleto,Aprovado,3,false,false,false,false,2026-05-02T05:56:31.184Z
000212a2-5abb-4fe7-a4bb-4c96de7fa25d,21688ea2-7f40-47f3-9c7f-8fee65eef675,PROD-0022,6458.47,2023-10-02,2023-10,Cartão,Aprovado,5,true,false,false,true,2026-05-02T05:56:31.184Z


In [0]:
# ─── Escrita na Silver ────────────────────────────────────────────────────────

(
    df_silver
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(f'{silver_schema}.ft_pedidos')
)

print(f'Tabela gravada com sucesso: {silver_schema}.ft_pedidos')

Tabela gravada com sucesso: vcommerce_silver.ft_pedidos


---

## Tratamento: `dim_status_pedido`

**Origem:** `bronze.pedidos`  
**Destino:** `silver.dim_status_pedido`

### Objetivo

Criar uma dimensão auxiliar com os status canônicos de pedidos, a partir dos valores encontrados na Bronze, mantendo consistência com a normalização aplicada na `silver.ft_pedidos`.

### Regras aplicadas

| Coluna | Lógica |
|---|---|
| `status` | Valor canônico obtido a partir do mapeamento das variações encontradas em `bronze.pedidos` |
| `status_normalizado` | Versão normalizada de `status`, com lowercase e remoção de espaços nas extremidades |

In [0]:
# ─── Criação da dimensão dim_status_pedido ────────────────────────────────────
# Cria a dimensão de status a partir dos valores encontrados na Bronze de pedidos.
# A mesma regra de mapeamento usada na ft_pedidos é reaplicada para garantir
# consistência entre a fato e a dimensão.

df_dim_status_pedido = (
    df_raw
    .withColumn('status', status_expr)
    .select('status')
    .distinct()
    .withColumn('status_normalizado', F.lower(F.trim(F.col('status').cast('string'))))
    .orderBy('status')
)

print('Dimensão dim_status_pedido montada.')
print(f'Registros na dimensão: {df_dim_status_pedido.count():,}')

Dimensão dim_status_pedido montada.
Registros na dimensão: 4


In [0]:
# ─── Validação: dim_status_pedido ─────────────────────────────────────────────
# Verifica os status canônicos presentes na dimensão.

display(df_dim_status_pedido)

status,status_normalizado
Aprovado,aprovado
Processando,processando
Recusado,recusado
Reembolsado,reembolsado


In [0]:
# ─── Escrita na Silver ────────────────────────────────────────────────────────

(
    df_dim_status_pedido
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(f'{silver_schema}.dim_status_pedido')
)

print(f'Tabela gravada com sucesso: {silver_schema}.dim_status_pedido')

Tabela gravada com sucesso: vcommerce_silver.dim_status_pedido
